# LandSenView geemap imagery explorer

Search and display **Sentinel-2**, **Landsat 4/5/7/8/9**, and
**Sentinel-1** imagery for any point or polygon. This is a focused Python
adaptation of `GEE/LandSenView_v2.js` for ArcGIS Pro notebooks, VS Code,
JupyterLab, and other Jupyter environments that support `ipywidgets`.

The notebook keeps the source script's core behavior:

- point AOIs use a configurable buffer; polygon AOIs use the drawn shape;
- date range, optical cloud filter, product level, and Sentinel-1 mode are configurable;
- optical scenes are grouped and mosaicked by UTC acquisition date;
- results can be selected by sensor, date, and the same band-combination presets;
- cloud masking is optional and affects display only.

**Security:** authentication uses Earth Engine's normal OAuth flow. Never
paste an API key, OAuth token, service-account key, or credential JSON into
this notebook. Earth Engine stores user credentials outside the notebook.
The project ID `hardy-tenure-383607` identifies the quota/billing project;
it is not an authentication secret.

**ArcGIS Pro compatibility:** imagery is always rendered in the interactive
geemap widget. Automatic ArcPy mirroring is disabled so searches also work in
VS Code and external kernels. When the notebook is running inside ArcGIS Pro,
the display controls can optionally add the selected mosaic to a chosen active
map or scene using `ArcGISProject("CURRENT")`.

### Quick workflow

1. Run every cell from the top.
2. Select **Point** and click the map, or select **Polygon** and draw one.
3. Set search options and click **Search imagery**.
4. Choose a sensor, output date, and visualization, then click **Add selected image**.
5. Optional: test the ArcGIS connection and mirror the mosaic into a map or 3D scene.


## 1. Setup

Use a cloned ArcGIS Pro conda environment rather than modifying the
default `arcgispro-py3` environment. In VS Code, select the Python
environment that contains Jupyter and the packages below.

If imports fail, uncomment and run the installation line once, restart
the kernel, and continue. Package installation is intentionally not
automatic.


In [ ]:
# Uncomment only when the selected environment does not already have these packages.
# %pip install -U geemap earthengine-api ipywidgets pandas


In [ ]:
from datetime import date, datetime, timedelta, timezone
from statistics import fmean
import html
import math
import threading
import traceback

import ee
import geemap
import geemap.common as geemap_common
import geemap.geemap as geemap_widget
import ipywidgets as widgets
import pandas as pd
from geemap import core as geemap_core
from IPython import get_ipython
from IPython.display import clear_output, display
from ipyleaflet import Marker

PROJECT_ID = "hardy-tenure-383607"

# geemap optionally mirrors layers and zoom operations into an ArcGIS Pro map
# whenever arcpy is loaded. Some ArcGIS notebook/IDE contexts expose arcpy but
# cannot resolve arcpy.mp.ArcGISProject("CURRENT"), causing OSError: CURRENT.
# This notebook deliberately displays imagery in its portable ipyleaflet map.
geemap_common.is_arcpy = lambda: False
geemap_widget.is_arcpy = lambda: False

# ArcGIS Pro resolves ArcGISProject('CURRENT') only in the application/
# notebook execution context. Widget callbacks can run on a Dummy-* worker
# thread, so capture both the kernel loop and the project while this setup
# cell is running. ArcPy work is dispatched back to this thread below.
NOTEBOOK_THREAD_ID = threading.get_ident()
try:
    NOTEBOOK_IO_LOOP = get_ipython().kernel.io_loop
except Exception:
    NOTEBOOK_IO_LOOP = None

ARCGIS_CONTEXT = {"arcpy": None, "project": None, "error": None}


def _capture_current_arcgis_project():
    if threading.get_ident() != NOTEBOOK_THREAD_ID:
        raise RuntimeError(
            "The ArcGIS Pro project must be captured on the notebook "
            "execution thread. Restart the kernel and run all cells."
        )
    try:
        import arcpy
    except ImportError as error:
        ARCGIS_CONTEXT.update(
            arcpy=None,
            project=None,
            error=f"ArcPy is unavailable: {error}",
        )
        return

    try:
        project = arcpy.mp.ArcGISProject("CURRENT")
    except Exception as error:
        ARCGIS_CONTEXT.update(
            arcpy=arcpy,
            project=None,
            error=f"{type(error).__name__}: {error}",
        )
        return

    ARCGIS_CONTEXT.update(arcpy=arcpy, project=project, error=None)


_capture_current_arcgis_project()

try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    print("Earth Engine credentials were not available. Starting OAuth authentication...")
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

print(f"Earth Engine initialized with project: {PROJECT_ID}")
print("Portable notebook-map mode enabled (automatic ArcPy mirroring disabled).")
if ARCGIS_CONTEXT["project"] is not None:
    print("ArcGIS Pro bridge captured on the notebook execution thread.")
else:
    print("ArcGIS Pro bridge is optional and was not captured in this kernel.")


## 2. Choose the area of interest

- **Point:** choose Point mode and click the map. The search uses the
  configured buffer around the clicked location.
- **Polygon:** choose Polygon mode and use the polygon or rectangle tool
  on the map. The most recently drawn geometry is used directly.

Click **Clear AOI** before switching to a completely new location.


In [ ]:
MAP = geemap.Map(center=(0, 0), zoom=2, height="620px")
try:
    MAP.add_basemap("HYBRID")
except Exception:
    pass
try:
    MAP.add("layer_manager")
except Exception:
    pass

aoi_mode = widgets.ToggleButtons(
    options=[("Point (click map)", "point"), ("Polygon (draw tool)", "polygon")],
    value="point",
    description="AOI mode:",
)
aoi_status = widgets.HTML(
    "<b>AOI:</b> Point mode is active. Click once on the map."
)
clear_aoi_button = widgets.Button(
    description="Clear AOI", icon="trash", button_style=""
)

POINT_STATE = {"coordinates": None, "marker": None}


def _map_add_widget_layer(layer):
    try:
        MAP.add(layer)
    except Exception:
        MAP.add_layer(layer)


def _map_remove_widget_layer(layer):
    try:
        MAP.remove(layer)
    except Exception:
        try:
            MAP.remove_layer(layer)
        except Exception:
            pass


def _clear_point_marker():
    marker = POINT_STATE.get("marker")
    if marker is not None:
        _map_remove_widget_layer(marker)
    POINT_STATE["coordinates"] = None
    POINT_STATE["marker"] = None


def _handle_map_interaction(**event):
    if event.get("type") != "click" or aoi_mode.value != "point":
        return

    coordinates = event.get("coordinates")
    if not coordinates or len(coordinates) != 2:
        return

    latitude, longitude = map(float, coordinates)
    POINT_STATE["coordinates"] = (longitude, latitude)

    marker = POINT_STATE.get("marker")
    if marker is None:
        marker = Marker(
            location=(latitude, longitude),
            draggable=False,
            title="LandSenView point AOI",
        )
        POINT_STATE["marker"] = marker
        _map_add_widget_layer(marker)
    else:
        marker.location = (latitude, longitude)

    aoi_status.value = (
        f"<b>AOI:</b> point at {latitude:.6f}, {longitude:.6f}. "
        "Set the buffer below, then search."
    )


def _clear_drawings():
    try:
        MAP.remove_drawn_features()
        return
    except Exception:
        pass

    draw_control = getattr(MAP, "draw_control", None)
    if draw_control is not None:
        for method_name in ("reset", "clear"):
            method = getattr(draw_control, method_name, None)
            if callable(method):
                method()
                break


def _clear_aoi(_=None):
    _clear_point_marker()
    _clear_drawings()
    try:
        MAP.remove_ee_layer("Query AOI")
    except Exception:
        pass
    if aoi_mode.value == "point":
        aoi_status.value = "<b>AOI:</b> Click the map to choose a point."
    else:
        aoi_status.value = (
            "<b>AOI:</b> Use the polygon or rectangle drawing tool."
        )


def _on_aoi_mode_change(change):
    if change.get("name") != "value":
        return
    if change["new"] == "point":
        aoi_status.value = (
            "<b>AOI:</b> Point mode is active. Click once on the map."
        )
    else:
        aoi_status.value = (
            "<b>AOI:</b> Polygon mode is active. Draw a polygon or rectangle."
        )


MAP.on_interaction(_handle_map_interaction)
aoi_mode.observe(_on_aoi_mode_change, names="value")
clear_aoi_button.on_click(_clear_aoi)

display(widgets.HBox([aoi_mode, clear_aoi_button]), aoi_status)
display(MAP)


## 3. Search and display options

The default date behavior mirrors the JavaScript tool: search backward
from today. Enable custom dates to provide an inclusive start/end range.
Optical cloud percentage is a metadata filter; the separate cloud-mask
option is applied only when an image is displayed.


In [ ]:
today_utc = datetime.now(timezone.utc).date()

buffer_km = widgets.BoundedFloatText(
    value=5.0, min=0.1, max=100.0, step=0.5, description="Buffer (km):"
)
use_custom_dates = widgets.Checkbox(
    value=False, description="Use custom date range"
)
lookback_days = widgets.BoundedIntText(
    value=30, min=1, max=3650, step=1, description="Lookback days:"
)
start_date = widgets.DatePicker(
    value=today_utc - timedelta(days=30), description="Start date:"
)
end_date = widgets.DatePicker(value=today_utc, description="End date:")
cloud_max = widgets.IntSlider(
    value=30,
    min=0,
    max=100,
    step=1,
    description="Max cloud %:",
    continuous_update=False,
)
max_dates = widgets.BoundedIntText(
    value=30, min=1, max=400, step=1, description="Max dates:"
)
s2_level = widgets.Dropdown(
    options=[("L1C (TOA)", "L1C"), ("L2A (SR)", "L2A")],
    value="L1C",
    description="Sentinel-2:",
)
landsat_level = widgets.Dropdown(
    options=[("TOA", "TOA"), ("L2 (SR)", "L2SR")],
    value="TOA",
    description="Landsat:",
)
s1_mode = widgets.Dropdown(
    options=["ANY", "IW", "EW", "SM", "WV"],
    value="ANY",
    description="S1 mode:",
)
cloud_mask_display = widgets.Checkbox(
    value=False, description="Cloud removal (display only)"
)


def _sync_date_controls(change=None):
    custom = use_custom_dates.value
    start_date.disabled = not custom
    end_date.disabled = not custom
    lookback_days.disabled = custom


use_custom_dates.observe(_sync_date_controls, names="value")
_sync_date_controls()


## 4. Visualization presets and image preparation

These preset names and band combinations follow `LandSenView_v2.js`.
Landsat scenes are first normalized to semantic band names so a single
preset works across Landsat 4/5/7 and Landsat 8/9.


In [ ]:
S2_COMPOSITES = {
    "True Color (4,3,2)": {
        "type": "rgb", "bands": ["B4", "B3", "B2"]
    },
    "Highlight Optimized Natural Color (4,3,2)": {
        "type": "highlight_rgb", "bands": ["B4", "B3", "B2"]
    },
    "QuickFire - True Color + SWIR Hotspots": {
        "type": "quickfire", "bands": ["B2", "B3", "B4", "B11", "B12"]
    },
    "False Color (8,4,3)": {
        "type": "rgb", "bands": ["B8", "B4", "B3"]
    },
    "SWIR (12,8,4)": {
        "type": "rgb", "bands": ["B12", "B8", "B4"]
    },
    "Agriculture (11,8,2)": {
        "type": "rgb", "bands": ["B11", "B8", "B2"]
    },
    "Geology (12,11,2)": {
        "type": "rgb", "bands": ["B12", "B11", "B2"]
    },
    "Bathymetric (4,3,1)": {
        "type": "rgb", "bands": ["B4", "B3", "B1"]
    },
    "RGB (8,6,4)": {
        "type": "rgb", "bands": ["B8", "B6", "B4"]
    },
    "RGB (8,5,4)": {
        "type": "rgb", "bands": ["B8", "B5", "B4"]
    },
    "RGB (8,11,4)": {
        "type": "rgb", "bands": ["B8", "B11", "B4"]
    },
    "RGB (8,11,12)": {
        "type": "rgb", "bands": ["B8", "B11", "B12"]
    },
    "RGB (11,8,3)": {
        "type": "rgb", "bands": ["B11", "B8", "B3"]
    },
    "NDVI": {"type": "ndvi"},
}

LANDSAT_COMPOSITES = {
    "Natural Color (L8/9: 4,3,2 | L4-7: 3,2,1)": {
        "type": "rgb", "bands": ["RED", "GREEN", "BLUE"]
    },
    "Highlight Optimized Natural Color (L8/9: 4,3,2 | L4-7: 3,2,1)": {
        "type": "highlight_rgb", "bands": ["RED", "GREEN", "BLUE"]
    },
    "QuickFire - Natural Color + SWIR Hotspots (L4/5/7/8/9)": {
        "type": "quickfire",
        "bands": ["BLUE", "GREEN", "RED", "SWIR1", "SWIR2"],
    },
    "Color Infrared (L8/9: 5,4,3 | L4-7: 4,3,2)": {
        "type": "rgb", "bands": ["NIR", "RED", "GREEN"]
    },
    "False Color (Urban) (L8/9: 7,6,4 | L4-7: 7,5,3)": {
        "type": "rgb", "bands": ["SWIR2", "SWIR1", "RED"]
    },
    "Agriculture (L8/9: 6,5,2 | L4-7: 5,4,1)": {
        "type": "rgb", "bands": ["SWIR1", "NIR", "BLUE"]
    },
    "Geology (L8/9: 7,6,2 | L4-7: 7,5,1)": {
        "type": "rgb", "bands": ["SWIR2", "SWIR1", "BLUE"]
    },
    "Atmospheric Penetration (L8/9: 7,6,5 | L4-7: 7,5,4)": {
        "type": "rgb", "bands": ["SWIR2", "SWIR1", "NIR"]
    },
    "Healthy Vegetation (L8/9: 5,6,2 | L4-7: 4,5,1)": {
        "type": "rgb", "bands": ["NIR", "SWIR1", "BLUE"]
    },
    "Land/Water (L8/9: 5,6,4 | L4-7: 4,5,3)": {
        "type": "rgb", "bands": ["NIR", "SWIR1", "RED"]
    },
    "Shortwave Infrared (L8/9: 7,5,4 | L4-7: 7,4,3)": {
        "type": "rgb", "bands": ["SWIR2", "NIR", "RED"]
    },
    "Vegetation Analysis (L8/9: 6,5,4 | L4-7: 5,4,3)": {
        "type": "rgb", "bands": ["SWIR1", "NIR", "RED"]
    },
    "Bathymetric (L8/9: 4,3,1* | L4-7: 3,2,1)": {
        "type": "rgb", "bands": ["RED", "GREEN", "BLUE"]
    },
    "NDVI": {"type": "ndvi"},
}

S1_VISUALIZATIONS = {
    "Auto single (prefer VH)": {"kind": "auto_single"},
    "Auto RGB ratio (dual-pol)": {"kind": "auto_rgb_ratio"},
    "Band: VV": {"kind": "band", "band": "VV"},
    "Band: VH": {"kind": "band", "band": "VH"},
    "Band: HH": {"kind": "band", "band": "HH"},
    "Band: HV": {"kind": "band", "band": "HV"},
    "Index: cross - co (dB) (auto)": {"kind": "auto_cross_minus_co"},
    "NDPI (auto dual-pol)": {"kind": "auto_ndpi"},
    "RVI4S1-like (auto dual-pol)": {"kind": "auto_rvi"},
}


TEMPORAL_REDUCERS = {
    "Minimum": "min",
    "Mean": "mean",
    "Median": "median",
    "Maximum": "max",
    "Mode": "mode",
}

OPTICAL_VIS = {"min": 0.02, "max": 0.35, "gamma": 1.1}
HIGHLIGHT_VIS = {"min": 0, "max": 0.9, "gamma": 1}
QUICKFIRE_VIS = {"min": 0, "max": 1, "gamma": 1}
NDVI_VIS = {
    "min": 0,
    "max": 1,
    "palette": ["#8c510a", "#d8b365", "#f6e8c3",
                "#c7eae5", "#5ab4ac", "#01665e"],
}
S1_BAND_VIS = {"min": -25, "max": 0}
S1_DIFF_VIS = {
    "min": -12,
    "max": 6,
    "palette": ["#2c7bb6", "#abd9e9", "#ffffbf", "#fdae61", "#d7191c"],
}
S1_INDEX_VIS = {
    "min": 0,
    "max": 1,
    "palette": ["#8e0152", "#de77ae", "#f7f7f7", "#7fbc41", "#276419"],
}


In [ ]:
def mask_s2_l2a_scl(image):
    image = ee.Image(image)
    scl = image.select("SCL")
    mask = (
        scl.neq(3)
        .And(scl.neq(8))
        .And(scl.neq(9))
        .And(scl.neq(10))
        .And(scl.neq(11))
    )
    return image.updateMask(mask)


def mask_s2_l1c_qa60(image):
    image = ee.Image(image)
    qa = image.select("QA60")
    cloud_mask = 1 << 10
    cirrus_mask = 1 << 11
    mask = (
        qa.bitwiseAnd(cloud_mask)
        .eq(0)
        .And(qa.bitwiseAnd(cirrus_mask).eq(0))
    )
    return image.updateMask(mask)


def mask_landsat_clouds(image):
    image = ee.Image(image)
    qa = image.select("QA_PIXEL")
    mask = (
        qa.bitwiseAnd(1 << 1)
        .eq(0)
        .And(qa.bitwiseAnd(1 << 2).eq(0))
        .And(qa.bitwiseAnd(1 << 3).eq(0))
        .And(qa.bitwiseAnd(1 << 4).eq(0))
        .And(qa.bitwiseAnd(1 << 5).eq(0))
    )
    return image.updateMask(mask)


def scale_landsat_sr(image):
    image = ee.Image(image)
    optical = image.select("SR_B.").multiply(0.0000275).add(-0.2)
    return image.addBands(optical, None, True)


def landsat_spacecraft_id(image):
    image = ee.Image(image)
    properties = image.propertyNames()
    return ee.String(
        ee.Algorithms.If(
            properties.contains("SPACECRAFT_ID"),
            image.get("SPACECRAFT_ID"),
            ee.Algorithms.If(
                properties.contains("SATELLITE"),
                image.get("SATELLITE"),
                "LANDSAT_8",
            ),
        )
    )


def landsat_select_band(image, sensor_key, oli_band, tm_band):
    image = ee.Image(image)
    is_oli = ee.List(["LANDSAT_8", "LANDSAT_9"]).contains(
        landsat_spacecraft_id(image)
    )
    prefix = "SR_B" if sensor_key == "Landsat_L2SR" else "B"
    return ee.Image(
        ee.Algorithms.If(
            is_oli,
            image.select(f"{prefix}{oli_band}"),
            image.select(f"{prefix}{tm_band}"),
        )
    )


def landsat_to_common_bands(image, sensor_key):
    image = ee.Image(image)
    source = scale_landsat_sr(image) if sensor_key == "Landsat_L2SR" else image
    return ee.Image.cat(
        [
            landsat_select_band(source, sensor_key, 2, 1).rename("BLUE"),
            landsat_select_band(source, sensor_key, 3, 2).rename("GREEN"),
            landsat_select_band(source, sensor_key, 4, 3).rename("RED"),
            landsat_select_band(source, sensor_key, 5, 4).rename("NIR"),
            landsat_select_band(source, sensor_key, 6, 5).rename("SWIR1"),
            landsat_select_band(source, sensor_key, 7, 7).rename("SWIR2"),
        ]
    )


def apply_highlight_optimized_natural_color(image, bands, is_surface_reflectance):
    image = ee.Image(image)
    red = image.select(bands[0]).multiply(0.6)
    green = image.select(bands[1]).multiply(0.6)
    blue = image.select(bands[2]).multiply(0.6)

    if not is_surface_reflectance:
        red = red.subtract(0.035)
        green = green.subtract(0.035)
        blue = blue.subtract(0.035)

    return ee.Image.cat(
        [
            red.max(0).pow(1 / 3).rename("R"),
            green.max(0).pow(1 / 3).rename("G"),
            blue.max(0).pow(1 / 3).rename("B"),
        ]
    )


def apply_quickfire_visualization(common_image):
    common_image = ee.Image(common_image)
    brightness = 1.0
    saturation = 1.10
    stretch_min = 0.01
    stretch_max = 0.99
    hotspot_sensitivity = 1.0
    boost = 1.0
    avoidance_helper = 0.80

    blue = common_image.select("BLUE")
    green_band = common_image.select("GREEN")
    red_band = common_image.select("RED")
    swir1 = common_image.select("SWIR1")
    swir2 = common_image.select("SWIR2")

    natural_gamma = ee.Image.cat(
        [
            red_band.multiply(brightness).max(0).sqrt().rename("R"),
            green_band.multiply(brightness).max(0).sqrt().rename("G"),
            blue.multiply(brightness).max(0).sqrt().rename("B"),
        ]
    )
    natural = ee.Image.cat(
        [
            red_band.multiply(2.5 * brightness).rename("R"),
            green_band.multiply(2.5 * brightness).rename("G"),
            blue.multiply(2.5 * brightness).rename("B"),
        ]
    )
    urban = ee.Image.cat(
        [
            swir2.multiply(1.2 * brightness).max(0).sqrt().rename("R"),
            swir1.multiply(1.4 * brightness).max(0).sqrt().rename("G"),
            red_band.multiply(brightness).max(0).sqrt().rename("B"),
        ]
    )

    visualization = (
        urban.multiply(0.10)
        .add(natural.multiply(0.40))
        .add(natural_gamma.multiply(0.50))
    )

    channel_mean = visualization.reduce(ee.Reducer.mean())
    mean_rgb = ee.Image.cat(
        [
            channel_mean.rename("R"),
            channel_mean.rename("G"),
            channel_mean.rename("B"),
        ]
    )
    visualization = visualization.multiply(saturation).add(
        mean_rgb.multiply(1 - saturation)
    )
    visualization = visualization.subtract(stretch_min).divide(
        stretch_max - stretch_min
    )

    swir_sum = swir2.add(swir1)
    hotspot_mask = swir_sum.gt(1.00 / hotspot_sensitivity).And(
        blue.lt(avoidance_helper)
    )
    green_factor = (
        swir1.multiply(0)
        .where(swir_sum.gt(1.25 / hotspot_sensitivity), 0.10)
        .where(swir_sum.gt(1.50 / hotspot_sensitivity), 0.20)
        .where(swir_sum.gt(2.00 / hotspot_sensitivity), 0.50)
    )

    red = visualization.select("R").where(
        hotspot_mask,
        visualization.select("R").add(swir2.multiply(boost * 0.50)),
    )
    green = visualization.select("G").where(
        hotspot_mask,
        visualization.select("G").add(
            swir1.multiply(green_factor).multiply(boost)
        ),
    )

    return ee.Image.cat(
        [red.rename("R"), green.rename("G"), visualization.select("B").rename("B")]
    ).clamp(0, 1)


def _images_from_ids(image_ids):
    return ee.ImageCollection.fromImages([ee.Image(image_id) for image_id in image_ids])


def reduce_image_collection(collection, reducer_name=None):
    """Mosaic a collection or apply a supported temporal reducer."""
    collection = ee.ImageCollection(collection)
    if reducer_name is None:
        return collection.mosaic()
    if reducer_name == "min":
        return collection.min()
    if reducer_name == "mean":
        return collection.mean()
    if reducer_name == "median":
        return collection.median()
    if reducer_name == "max":
        return collection.max()
    if reducer_name == "mode":
        return collection.mode()
    raise ValueError(f"Unsupported temporal reducer: {reducer_name!r}")


def make_s2_display_image(
    record,
    preset_name,
    apply_cloud_mask,
    source_collection=None,
    temporal_reducer=None,
):
    sensor_key = record["sensor_key"]
    preset = S2_COMPOSITES[preset_name]
    collection = (
        ee.ImageCollection(source_collection)
        if source_collection is not None
        else _images_from_ids(record["ids"])
    )

    def prepare(image):
        image = ee.Image(image)
        if apply_cloud_mask:
            image = (
                mask_s2_l2a_scl(image)
                if sensor_key == "S2_L2A"
                else mask_s2_l1c_qa60(image)
            )
        if preset["type"] == "ndvi":
            return image.normalizedDifference(["B8", "B4"]).rename("NDVI")

        scaled = image.select(preset["bands"]).multiply(0.0001)
        if preset["type"] == "highlight_rgb":
            return apply_highlight_optimized_natural_color(
                scaled, preset["bands"], sensor_key == "S2_L2A"
            )
        if preset["type"] == "quickfire":
            common = scaled.select(
                ["B2", "B3", "B4", "B11", "B12"],
                ["BLUE", "GREEN", "RED", "SWIR1", "SWIR2"],
            )
            return apply_quickfire_visualization(common)
        return scaled

    image = reduce_image_collection(
        collection.map(prepare), temporal_reducer
    )
    if preset["type"] == "ndvi":
        return image, NDVI_VIS
    if preset["type"] == "highlight_rgb":
        return image, HIGHLIGHT_VIS
    if preset["type"] == "quickfire":
        return image, QUICKFIRE_VIS
    return image, OPTICAL_VIS


def make_landsat_display_image(
    record,
    preset_name,
    apply_cloud_mask,
    source_collection=None,
    temporal_reducer=None,
):
    sensor_key = record["sensor_key"]
    preset = LANDSAT_COMPOSITES[preset_name]
    collection = (
        ee.ImageCollection(source_collection)
        if source_collection is not None
        else _images_from_ids(record["ids"])
    )

    def prepare(image):
        image = ee.Image(image)
        if apply_cloud_mask:
            image = mask_landsat_clouds(image)
        common = landsat_to_common_bands(image, sensor_key)

        if preset["type"] == "ndvi":
            return common.normalizedDifference(["NIR", "RED"]).rename("NDVI")
        if preset["type"] == "highlight_rgb":
            return apply_highlight_optimized_natural_color(
                common, preset["bands"], sensor_key == "Landsat_L2SR"
            )
        if preset["type"] == "quickfire":
            return apply_quickfire_visualization(common)
        return common.select(preset["bands"])

    image = reduce_image_collection(
        collection.map(prepare), temporal_reducer
    )
    if preset["type"] == "ndvi":
        return image, NDVI_VIS
    if preset["type"] == "highlight_rgb":
        return image, HIGHLIGHT_VIS
    if preset["type"] == "quickfire":
        return image, QUICKFIRE_VIS
    return image, OPTICAL_VIS


def _db_to_linear(image):
    return ee.Image(10).pow(ee.Image(image).divide(10))


def _s1_auto_single(image, polarizations):
    for band in ("VH", "VV", "HV", "HH"):
        if band in polarizations:
            return image.select(band)
    return image.select([0])


def _s1_co_cross(polarizations):
    if "VV" in polarizations and "VH" in polarizations:
        return "VV", "VH"
    if "HH" in polarizations and "HV" in polarizations:
        return "HH", "HV"
    return None


def make_s1_display_image(
    record, preset_name, source_collection=None, temporal_reducer=None
):
    collection = (
        ee.ImageCollection(source_collection)
        if source_collection is not None
        else _images_from_ids(record["ids"])
    )
    image = reduce_image_collection(collection, temporal_reducer)
    polarizations = record.get("polarizations") or []
    preset = S1_VISUALIZATIONS[preset_name]
    kind = preset["kind"]

    if kind == "auto_single":
        return _s1_auto_single(image, polarizations), S1_BAND_VIS
    if kind == "band":
        if preset["band"] in polarizations:
            return image.select(preset["band"]), S1_BAND_VIS
        return _s1_auto_single(image, polarizations), S1_BAND_VIS

    co_cross = _s1_co_cross(polarizations)
    if co_cross is None:
        return _s1_auto_single(image, polarizations), S1_BAND_VIS

    co_band, cross_band = co_cross
    co = image.select(co_band)
    cross = image.select(cross_band)

    if kind == "auto_rgb_ratio":
        return (
            ee.Image.cat(
                [
                    co.rename("R"),
                    cross.rename("G"),
                    cross.subtract(co).rename("B"),
                ]
            ),
            S1_BAND_VIS,
        )
    if kind == "auto_cross_minus_co":
        return cross.subtract(co).rename("cross_minus_co_db"), S1_DIFF_VIS

    co_linear = _db_to_linear(co)
    cross_linear = _db_to_linear(cross)
    if kind == "auto_ndpi":
        ndpi = co_linear.subtract(cross_linear).divide(
            co_linear.add(cross_linear)
        )
        return ndpi.rename("NDPI"), {
            "min": -1,
            "max": 1,
            "palette": S1_DIFF_VIS["palette"],
        }

    ratio = cross_linear.divide(co_linear)
    rvi = ratio.multiply(ratio.add(3)).divide(ratio.add(1).pow(2))
    return rvi.rename("RVI4S1_like"), S1_INDEX_VIS


## 5. Query logic

Searches use the harmonized Sentinel-2 collections, Landsat Collection 2
Tier 1 collections, and `COPERNICUS/S1_GRD`. A client-side metadata step
reproduces the JavaScript tool's UTC-date grouping. To protect notebook
responsiveness, the metadata transfer is capped at the most recent 2,500
scenes per sensor; narrow the AOI/date range if a warning reports that cap.


In [ ]:
METADATA_SCENE_CAP = 2500
RESULTS = {"Sentinel-2": [], "Landsat": [], "Sentinel-1": []}
SEARCH_CONTEXT = {}
DISPLAYED_LAYERS = []
ARCGIS_LAYER_NAMES = []
# The most recently prepared display is kept for the ordinary ArcGIS Pro
# code cell below. Unlike widget callbacks, normal cell execution can use
# arcpy.mp.ArcGISProject("CURRENT") reliably.
LAST_DISPLAY_RESULT = {}


def _selected_geometry():
    if aoi_mode.value == "point":
        coordinates = POINT_STATE.get("coordinates")
        if coordinates is None:
            raise ValueError("Choose Point mode and click the map first.")
        longitude, latitude = coordinates
        point = ee.Geometry.Point([longitude, latitude])
        return point.buffer(float(buffer_km.value) * 1000), point

    geometry = getattr(MAP, "user_roi", None)
    if geometry is None:
        draw_control = getattr(MAP, "_draw_control", None)
        geometry = getattr(draw_control, "last_geometry", None)
    if geometry is None:
        raise ValueError(
            "Choose Polygon mode and draw a polygon or rectangle first."
        )
    return geometry, geometry


def _query_date_range():
    if use_custom_dates.value:
        if start_date.value is None or end_date.value is None:
            raise ValueError("Select both a custom start date and end date.")
        if start_date.value > end_date.value:
            raise ValueError("The start date must be before or equal to the end date.")
        start = start_date.value
        end_exclusive = end_date.value + timedelta(days=1)
        label = f"{start.isoformat()} to {end_date.value.isoformat()} (inclusive)"
    else:
        end_inclusive = datetime.now(timezone.utc).date()
        start = end_inclusive - timedelta(days=int(lookback_days.value))
        end_exclusive = end_inclusive + timedelta(days=1)
        label = (
            f"{start.isoformat()} to {end_inclusive.isoformat()} "
            f"({int(lookback_days.value)}-day lookback)"
        )
    return start.isoformat(), end_exclusive.isoformat(), label


def _merge_collections(collection_ids):
    collection = ee.ImageCollection(collection_ids[0])
    for collection_id in collection_ids[1:]:
        collection = collection.merge(ee.ImageCollection(collection_id))
    return collection


def _build_collections(query_geometry, start_iso, end_exclusive_iso):
    if s2_level.value == "L2A":
        s2_id = "COPERNICUS/S2_SR_HARMONIZED"
        s2_sensor_key = "S2_L2A"
    else:
        s2_id = "COPERNICUS/S2_HARMONIZED"
        s2_sensor_key = "S2_L1C"

    sentinel2 = (
        ee.ImageCollection(s2_id)
        .filterBounds(query_geometry)
        .filterDate(start_iso, end_exclusive_iso)
        .filter(ee.Filter.lte("CLOUDY_PIXEL_PERCENTAGE", int(cloud_max.value)))
        .sort("system:time_start", False)
    )

    suffix = "T1_L2" if landsat_level.value == "L2SR" else "T1_TOA"
    landsat_ids = [
        f"LANDSAT/LT04/C02/{suffix}",
        f"LANDSAT/LT05/C02/{suffix}",
        f"LANDSAT/LE07/C02/{suffix}",
        f"LANDSAT/LC08/C02/{suffix}",
        f"LANDSAT/LC09/C02/{suffix}",
    ]
    landsat = (
        _merge_collections(landsat_ids)
        .filterBounds(query_geometry)
        .filterDate(start_iso, end_exclusive_iso)
        .filter(ee.Filter.lte("CLOUD_COVER", int(cloud_max.value)))
        .sort("system:time_start", False)
    )

    sentinel1 = (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(query_geometry)
        .filterDate(start_iso, end_exclusive_iso)
        .sort("system:time_start", False)
    )
    if s1_mode.value != "ANY":
        sentinel1 = sentinel1.filter(
            ee.Filter.eq("instrumentMode", s1_mode.value)
        )

    return {
        "Sentinel-2": (sentinel2, s2_sensor_key),
        "Landsat": (
            landsat,
            "Landsat_L2SR" if landsat_level.value == "L2SR" else "Landsat_TOA",
        ),
        "Sentinel-1": (sentinel1, "S1"),
    }


def _collection_metadata(collection, property_names):
    total_scenes = int(collection.size().getInfo())
    limited = ee.ImageCollection(
        collection.limit(
            METADATA_SCENE_CAP, "system:time_start", False
        )
    )
    values = {
        "ids": limited.aggregate_array("system:id"),
        "times": limited.aggregate_array("system:time_start"),
    }
    for output_name, property_name in property_names.items():
        values[output_name] = limited.aggregate_array(property_name)

    payload = ee.Dictionary(values).getInfo()
    payload = payload or {}
    payload["total_scenes"] = total_scenes
    payload["truncated"] = total_scenes > METADATA_SCENE_CAP
    return payload


def _item(values, index, default=None):
    if values is None or index >= len(values):
        return default
    value = values[index]
    return default if value is None else value


def _utc_date(milliseconds):
    return datetime.fromtimestamp(
        float(milliseconds) / 1000, tz=timezone.utc
    ).date().isoformat()


def _utc_time_range(start_milliseconds, end_milliseconds):
    start_text = datetime.fromtimestamp(
        start_milliseconds / 1000, tz=timezone.utc
    ).strftime("%H:%M")
    end_text = datetime.fromtimestamp(
        end_milliseconds / 1000, tz=timezone.utc
    ).strftime("%H:%M")
    return start_text if start_text == end_text else f"{start_text}-{end_text}"


def _group_optical(payload, sensor_key, cloud_property, include_missions=False):
    grouped = {}
    ids = payload.get("ids") or []
    times = payload.get("times") or []
    clouds = payload.get(cloud_property) or []
    spacecraft = payload.get("spacecraft") or []

    for index, image_id in enumerate(ids):
        milliseconds = float(_item(times, index))
        day = _utc_date(milliseconds)
        group = grouped.setdefault(
            day,
            {
                "date": day,
                "ids": [],
                "clouds": [],
                "missions": set(),
                "time_min": milliseconds,
                "time_max": milliseconds,
                "sensor_key": sensor_key,
            },
        )
        group["ids"].append(image_id)
        group["time_min"] = min(group["time_min"], milliseconds)
        group["time_max"] = max(group["time_max"], milliseconds)

        cloud_value = _item(clouds, index)
        if cloud_value is not None:
            group["clouds"].append(float(cloud_value))
        if include_missions:
            mission = _item(spacecraft, index)
            if mission:
                group["missions"].add(str(mission))

    records = []
    for day in sorted(grouped, reverse=True)[: int(max_dates.value)]:
        group = grouped[day]
        records.append(
            {
                "date": day,
                "ids": group["ids"],
                "tile_count": len(group["ids"]),
                "mean_cloud": (
                    fmean(group["clouds"]) if group["clouds"] else None
                ),
                "missions": sorted(group["missions"]),
                "time_utc": _utc_time_range(
                    group["time_min"], group["time_max"]
                ),
                "sensor_key": sensor_key,
            }
        )
    return records


def _group_sentinel1(payload):
    grouped = {}
    ids = payload.get("ids") or []
    times = payload.get("times") or []
    passes = payload.get("passes") or []
    relative_orbits = payload.get("relative_orbits") or []
    modes = payload.get("modes") or []
    polarizations = payload.get("polarizations") or []

    for index, image_id in enumerate(ids):
        milliseconds = float(_item(times, index))
        day = _utc_date(milliseconds)
        group = grouped.setdefault(
            day,
            {
                "date": day,
                "ids": [],
                "passes": [],
                "relative_orbits": [],
                "modes": [],
                "polarizations": [],
                "time_min": milliseconds,
                "time_max": milliseconds,
                "sensor_key": "S1",
            },
        )
        group["ids"].append(image_id)
        group["time_min"] = min(group["time_min"], milliseconds)
        group["time_max"] = max(group["time_max"], milliseconds)
        group["passes"].append(_item(passes, index))
        group["relative_orbits"].append(_item(relative_orbits, index))
        group["modes"].append(_item(modes, index))
        group["polarizations"].append(_item(polarizations, index, []))

    records = []
    for day in sorted(grouped, reverse=True)[: int(max_dates.value)]:
        group = grouped[day]
        records.append(
            {
                "date": day,
                "ids": group["ids"],
                "tile_count": len(group["ids"]),
                "pass": next(
                    (value for value in group["passes"] if value is not None),
                    None,
                ),
                "relative_orbit": next(
                    (
                        value
                        for value in group["relative_orbits"]
                        if value is not None
                    ),
                    None,
                ),
                "mode": next(
                    (value for value in group["modes"] if value is not None),
                    None,
                ),
                "polarizations": next(
                    (
                        list(value)
                        for value in group["polarizations"]
                        if value
                    ),
                    [],
                ),
                "time_utc": _utc_time_range(
                    group["time_min"], group["time_max"]
                ),
                "sensor_key": "S1",
            }
        )
    return records


def _query_all_sensors(query_geometry, start_iso, end_exclusive_iso):
    collections = _build_collections(
        query_geometry, start_iso, end_exclusive_iso
    )

    s2_collection, s2_key = collections["Sentinel-2"]
    s2_payload = _collection_metadata(
        s2_collection, {"clouds": "CLOUDY_PIXEL_PERCENTAGE"}
    )
    RESULTS["Sentinel-2"] = _group_optical(
        s2_payload, s2_key, "clouds"
    )

    landsat_collection, landsat_key = collections["Landsat"]
    landsat_payload = _collection_metadata(
        landsat_collection,
        {"clouds": "CLOUD_COVER", "spacecraft": "SPACECRAFT_ID"},
    )
    RESULTS["Landsat"] = _group_optical(
        landsat_payload,
        landsat_key,
        "clouds",
        include_missions=True,
    )

    s1_collection, _ = collections["Sentinel-1"]
    s1_payload = _collection_metadata(
        s1_collection,
        {
            "passes": "orbitProperties_pass",
            "relative_orbits": "relativeOrbitNumber_start",
            "modes": "instrumentMode",
            "polarizations": "transmitterReceiverPolarisation",
        },
    )
    RESULTS["Sentinel-1"] = _group_sentinel1(s1_payload)

    payloads = {
        "Sentinel-2": s2_payload,
        "Landsat": landsat_payload,
        "Sentinel-1": s1_payload,
    }
    return payloads, collections


def _selected_date_collection(sensor, record):
    """Return every filtered granule/scene for the selected UTC date."""
    collection_entry = SEARCH_CONTEXT.get("collections", {}).get(sensor)
    if collection_entry is None:
        return _images_from_ids(record["ids"])

    base_collection, _sensor_key = collection_entry
    day_start = ee.Date(record["date"])
    day_end = day_start.advance(1, "day")
    return ee.ImageCollection(base_collection).filterDate(day_start, day_end)



def _search_range_collection(sensor):
    """Return the complete filtered collection from the latest search."""
    collection_entry = SEARCH_CONTEXT.get("collections", {}).get(sensor)
    if collection_entry is None:
        raise RuntimeError("Search first to create a date-range collection.")
    return ee.ImageCollection(collection_entry[0])


def _search_range_record(sensor):
    """Build the metadata context needed to visualize a range reducer."""
    records = RESULTS.get(sensor, [])
    if not records:
        raise RuntimeError(f"No {sensor} images were returned by the search.")

    record = dict(records[0])
    collection_entry = SEARCH_CONTEXT["collections"][sensor]
    record["sensor_key"] = collection_entry[1]
    if sensor == "Sentinel-1":
        record["polarizations"] = sorted(
            {
                polarization
                for item in records
                for polarization in item.get("polarizations", [])
            }
        )
    return record


def _search_range_dates():
    start_iso = SEARCH_CONTEXT["start"]
    end_inclusive = (
        date.fromisoformat(SEARCH_CONTEXT["end_exclusive"])
        - timedelta(days=1)
    ).isoformat()
    return start_iso, end_inclusive


## 6. Interactive explorer

Search results appear as compact tables. The selection controls then let
you add any returned date and any compatible visualization to the map.
Use the map's layer manager to toggle visibility and opacity.


In [ ]:
search_button = widgets.Button(
    description="Search imagery", icon="search", button_style="primary"
)
clear_results_button = widgets.Button(
    description="Clear results", icon="eraser"
)
search_status = widgets.HTML("<b>Status:</b> Ready.")
search_output = widgets.Output()

sensor_select = widgets.Dropdown(
    options=list(RESULTS), value="Sentinel-2", description="Sensor:"
)
date_select = widgets.Dropdown(options=[], description="Output:")
visualization_select = widgets.Dropdown(
    options=list(S2_COMPOSITES), description="Composite:"
)
temporal_mode = widgets.Dropdown(
    options=[
        ("Selected date mosaic", "date"),
        ("Search-range mosaic (latest on top)", "range_mosaic"),
        ("Search-range reducer", "range_reducer"),
    ],
    value="date",
    description="Temporal:",
    layout=widgets.Layout(width="360px"),
)
temporal_reducer = widgets.Dropdown(
    options=[(label, value) for label, value in TEMPORAL_REDUCERS.items()],
    value="median",
    description="Reducer:",
    layout=widgets.Layout(width="300px"),
)
replace_previous = widgets.Checkbox(
    value=True, description="Replace previously displayed result layers"
)
zoom_to_result = widgets.Checkbox(
    value=True, description="Zoom to query AOI"
)
mirror_to_arcgis = widgets.Checkbox(
    value=False, description="Also add mosaic to ArcGIS Pro"
)
open_arcgis_view = widgets.Checkbox(
    value=False,
    description="Adjust existing active Scene camera",
)
arcgis_target = widgets.Dropdown(
    options=[("Active map/scene", "__ACTIVE__")],
    value="__ACTIVE__",
    description="ArcGIS target:",
    layout=widgets.Layout(width="430px"),
)
test_arcgis_button = widgets.Button(
    description="Test ArcGIS connection", icon="check"
)
arcgis_status = widgets.HTML(
    "<b>ArcGIS Pro:</b> Optional. Test the connection before mirroring."
)
add_layer_button = widgets.Button(
    description="Add selected image", icon="plus", button_style="success"
)
clear_layers_button = widgets.Button(
    description="Clear image layers", icon="trash"
)
display_status = widgets.HTML("<b>Display:</b> Search first.")


def _safe_remove_ee_layer(layer_name):
    try:
        MAP.remove_ee_layer(layer_name)
    except Exception:
        pass


def _add_ee_layer(ee_object, vis_params, layer_name, shown=True):
    """Add an EE layer only to the portable notebook map."""
    geemap_core.Map.add_layer(
        MAP, ee_object, vis_params, layer_name, shown
    )


def _center_object(ee_object, zoom=None):
    """Center the notebook map without ipyleaflet's asyncio fit_bounds()."""
    if isinstance(ee_object, ee.Geometry):
        geometry = ee_object
    else:
        geometry = ee_object.geometry(maxError=1)

    coordinates = geometry.bounds(maxError=1).getInfo()["coordinates"][0]
    x_values = [coordinate[0] for coordinate in coordinates]
    y_values = [coordinate[1] for coordinate in coordinates]
    xmin, xmax = min(x_values), max(x_values)
    ymin, ymax = min(y_values), max(y_values)

    MAP.center = ((ymin + ymax) / 2, (xmin + xmax) / 2)
    if zoom is None:
        longitude_span = max(xmax - xmin, 1e-6)
        latitude_span = max(ymax - ymin, 1e-6)
        display_span = max(longitude_span, latitude_span * 1.6)
        zoom = math.floor(math.log2(360 / display_span))
        zoom = max(1, min(18, zoom))
    MAP.zoom = int(zoom)


def _run_on_notebook_thread(operation, timeout_seconds=55):
    """Run an ArcPy operation in the thread that executed the setup cell."""
    if threading.get_ident() == NOTEBOOK_THREAD_ID:
        return operation()
    if NOTEBOOK_IO_LOOP is None:
        raise RuntimeError(
            "ArcGIS Pro cannot run ArcPy from this background widget thread. "
            "The selected mosaic is still prepared; run the "
            "'Send the prepared layer to ArcGIS Pro' code cell below."
        )

    completed = threading.Event()
    outcome = {}

    def invoke():
        try:
            outcome["result"] = operation()
        except BaseException as error:
            outcome["error"] = error
        finally:
            completed.set()

    NOTEBOOK_IO_LOOP.add_callback(invoke)
    if not completed.wait(timeout_seconds):
        raise TimeoutError(
            "ArcGIS Pro did not complete the notebook-thread operation within "
            f"{timeout_seconds} seconds."
        )
    if "error" in outcome:
        raise outcome["error"]
    return outcome.get("result")


def _arcgis_project():
    """Return the project captured during cell execution, never from a worker."""
    if ARCGIS_CONTEXT["project"] is None:
        if threading.get_ident() == NOTEBOOK_THREAD_ID:
            _capture_current_arcgis_project()
        if ARCGIS_CONTEXT["project"] is None:
            detail = ARCGIS_CONTEXT["error"] or "unknown initialization error"
            raise RuntimeError(
                "The current ArcGIS Pro project was not captured while the "
                f"setup cell was running ({detail}). In ArcGIS Pro, restart "
                "this notebook's kernel and run all cells. An external "
                "VS Code/Jupyter kernel cannot modify the open Pro scene."
            )
    return ARCGIS_CONTEXT["arcpy"], ARCGIS_CONTEXT["project"]


def _refresh_arcgis_targets():
    def operation():
        _arcpy, project = _arcgis_project()
        maps = project.listMaps()
        if not maps:
            raise RuntimeError(
                "The current ArcGIS Pro project has no maps or scenes."
            )

        options = [(f"{item.name} ({item.mapType})", item.name) for item in maps]
        active_map = project.activeMap
        preferred = active_map.name if active_map is not None else None
        if preferred is None:
            preferred = next(
                (item.name for item in maps if item.mapType == "SCENE"),
                maps[0].name,
            )
        return {
            "options": options,
            "preferred": preferred,
            "map_count": len(maps),
            "scene_count": sum(item.mapType == "SCENE" for item in maps),
        }

    return _run_on_notebook_thread(operation)


def _apply_arcgis_targets(payload):
    arcgis_target.options = payload["options"]
    arcgis_target.value = payload["preferred"]


def _on_test_arcgis(_):
    test_arcgis_button.disabled = True
    if (
        NOTEBOOK_IO_LOOP is None
        and threading.get_ident() != NOTEBOOK_THREAD_ID
    ):
        arcgis_status.value = (
            "<b>ArcGIS Pro:</b> The map/scene list was loaded when this UI "
            "cell ran. Rerun the UI cell to refresh it; use the ordinary "
            "ArcGIS Pro transfer cell below to add the prepared mosaic."
        )
        test_arcgis_button.disabled = False
        return
    try:
        payload = _refresh_arcgis_targets()
        _apply_arcgis_targets(payload)
        arcgis_status.value = (
            f"<b>ArcGIS Pro:</b> Connected — {payload['map_count']} map(s), "
            f"{payload['scene_count']} scene(s). Choose a target above."
        )
    except Exception as error:
        arcgis_status.value = (
            f"<b>ArcGIS Pro:</b> <span style='color:#b00020'>"
            f"{html.escape(str(error))}</span>"
        )
    finally:
        test_arcgis_button.disabled = False


def _arcgis_target_map(project, selected_name):
    if selected_name == "__ACTIVE__":
        target_map = project.activeMap
        if target_map is None:
            raise RuntimeError(
                "No active ArcGIS map or scene. Use Test ArcGIS connection "
                "and choose a target."
            )
        return target_map

    matches = project.listMaps(selected_name)
    if not matches:
        raise RuntimeError(f"ArcGIS target {selected_name!r} no longer exists.")
    return matches[0]


def _earth_engine_tile_url(layer_name):
    layer_info = MAP.ee_layers.get(layer_name, {})
    tile_layer = layer_info.get("ee_layer")
    if tile_layer is None:
        raise RuntimeError("The notebook Earth Engine tile layer was not created.")
    url = getattr(tile_layer, "url_format", None) or getattr(tile_layer, "url", None)
    if not url:
        raise RuntimeError("Could not obtain the Earth Engine tile-service URL.")
    return url


def _geometry_extent(geometry):
    coordinates = geometry.bounds(maxError=1).getInfo()["coordinates"][0]
    x_values = [coordinate[0] for coordinate in coordinates]
    y_values = [coordinate[1] for coordinate in coordinates]
    return min(x_values), min(y_values), max(x_values), max(y_values)


def _arcgis_add_layer(layer_name, focus_geometry):
    tile_url = _earth_engine_tile_url(layer_name)
    bounds = _geometry_extent(focus_geometry)
    selected_name = arcgis_target.value
    should_open_view = bool(open_arcgis_view.value)

    def operation():
        arcpy, project = _arcgis_project()
        target_map = _arcgis_target_map(project, selected_name)

        # Scene maps can contain special Layer proxies (for example,
        # elevation/3D layers) that do not expose the normal .name property.
        # Ask ArcGIS to perform the name filtering instead of reading .name
        # from every layer in the scene.
        try:
            matching_layers = list(target_map.listLayers(layer_name))
        except Exception:
            matching_layers = []
        for existing_layer in matching_layers:
            try:
                target_map.removeLayer(existing_layer)
            except Exception:
                pass

        added_layer = target_map.addDataFromPath(tile_url)
        if added_layer is not None:
            try:
                added_layer.name = layer_name
            except (AttributeError, RuntimeError):
                # Naming is cosmetic; keep the successfully added tile layer
                # when this ArcGIS layer subtype exposes no writable name.
                pass

        xmin, ymin, xmax, ymax = bounds
        extent = arcpy.Extent(
            xmin,
            ymin,
            xmax,
            ymax,
            spatial_reference=arcpy.SpatialReference(4326),
        )
        target_map.defaultCamera.setExtent(extent)

        if should_open_view:
            # Never call target_map.openView(): ArcGIS Pro creates another tab
            # even when a view of this Scene is already open. Only adjust the
            # camera when the existing active map/view is already the target.
            active_map = project.activeMap
            active_view = project.activeView
            try:
                target_is_active = (
                    active_map is not None
                    and active_map.name == target_map.name
                )
            except (AttributeError, RuntimeError):
                target_is_active = False
            if (
                target_is_active
                and active_view is not None
                and hasattr(active_view, "camera")
            ):
                active_view.camera.setExtent(extent)

        return f"{target_map.name} ({target_map.mapType})"

    target_label = _run_on_notebook_thread(operation)
    if layer_name not in ARCGIS_LAYER_NAMES:
        ARCGIS_LAYER_NAMES.append(layer_name)
    return target_label


def _remove_arcgis_result_layers():
    if not ARCGIS_LAYER_NAMES:
        return
    names = set(ARCGIS_LAYER_NAMES)

    def operation():
        _arcpy, project = _arcgis_project()
        for project_map in project.listMaps():
            for layer_name in names:
                try:
                    matching_layers = list(project_map.listLayers(layer_name))
                except Exception:
                    matching_layers = []
                for layer in matching_layers:
                    try:
                        project_map.removeLayer(layer)
                    except Exception:
                        pass

    try:
        _run_on_notebook_thread(operation)
    except Exception:
        pass
    finally:
        ARCGIS_LAYER_NAMES.clear()


def add_prepared_result_to_arcgis():
    """Add the cached result from an ordinary ArcGIS Pro notebook cell."""
    if threading.get_ident() != NOTEBOOK_THREAD_ID:
        raise RuntimeError(
            "Run this function from an ordinary notebook code cell, not from "
            "an ipywidgets callback."
        )
    if not LAST_DISPLAY_RESULT:
        raise RuntimeError(
            "No result is prepared. Search, choose an output, click "
            "'Add selected image', and then rerun the ArcGIS Pro cell below."
        )

    # If setup could not choose a target, resolve and select one now while
    # this ordinary cell has access to ArcGISProject("CURRENT").
    if arcgis_target.value == "__ACTIVE__":
        _apply_arcgis_targets(_refresh_arcgis_targets())

    layer_name = LAST_DISPLAY_RESULT["layer_name"]
    focus_geometry = LAST_DISPLAY_RESULT["focus_geometry"]
    target_label = _arcgis_add_layer(layer_name, focus_geometry)
    arcgis_status.value = (
        f"<b>ArcGIS Pro:</b> Added {html.escape(layer_name)} "
        f"to {html.escape(target_label)} from the notebook cell."
    )
    return target_label


def _record_label(sensor, record):
    if sensor == "Sentinel-2":
        cloud = record["mean_cloud"]
        cloud_text = "n/a" if cloud is None else f"{cloud:.1f}%"
        return (
            f"{record['date']} | {record['time_utc']} UTC | "
            f"{record['tile_count']} tiles | cloud {cloud_text}"
        )
    if sensor == "Landsat":
        cloud = record["mean_cloud"]
        cloud_text = "n/a" if cloud is None else f"{cloud:.1f}%"
        missions = ",".join(
            mission.replace("LANDSAT_", "L")
            for mission in record["missions"]
        ) or "L?"
        return (
            f"{record['date']} | {record['time_utc']} UTC | "
            f"{record['tile_count']} scenes | {missions} | cloud {cloud_text}"
        )

    polarizations = ",".join(record["polarizations"]) or "n/a"
    return (
        f"{record['date']} | {record['time_utc']} UTC | "
        f"{record['tile_count']} scenes | {record['mode'] or 'n/a'} | "
        f"{polarizations} | {record['pass'] or 'n/a'} | "
        f"orbit {record['relative_orbit'] or 'n/a'}"
    )


def _result_table(sensor):
    rows = []
    for record in RESULTS[sensor]:
        row = {
            "date": record["date"],
            "time_utc": record["time_utc"],
            "scenes_or_tiles": record["tile_count"],
        }
        if sensor in ("Sentinel-2", "Landsat"):
            row["mean_cloud_percent"] = (
                None
                if record["mean_cloud"] is None
                else round(record["mean_cloud"], 1)
            )
        if sensor == "Landsat":
            row["missions"] = ", ".join(record["missions"])
        if sensor == "Sentinel-1":
            row.update(
                {
                    "mode": record["mode"],
                    "polarizations": ", ".join(record["polarizations"]),
                    "pass": record["pass"],
                    "relative_orbit": record["relative_orbit"],
                }
            )
        rows.append(row)
    return pd.DataFrame(rows)


def _update_date_options():
    sensor = sensor_select.value
    records = RESULTS.get(sensor, [])
    date_select.options = [
        (_record_label(sensor, record), record["date"])
        for record in records
    ]
    if records:
        date_select.value = records[0]["date"]


def _update_visualization_options():
    sensor = sensor_select.value
    if sensor == "Sentinel-2":
        options = list(S2_COMPOSITES)
        visualization_select.description = "Composite:"
    elif sensor == "Landsat":
        options = list(LANDSAT_COMPOSITES)
        visualization_select.description = "Composite:"
    else:
        options = list(S1_VISUALIZATIONS)
        visualization_select.description = "Radar viz:"
    visualization_select.options = options
    visualization_select.value = options[0]



def _sync_temporal_controls(change=None):
    use_search_range = temporal_mode.value in {
        "range_mosaic",
        "range_reducer",
    }
    date_select.disabled = use_search_range
    temporal_reducer.disabled = temporal_mode.value != "range_reducer"


def _on_sensor_change(change):
    if change.get("name") == "value":
        _update_date_options()
        _update_visualization_options()


def _render_search_tables(metadata_payloads, date_label):
    with search_output:
        clear_output(wait=True)
        print(f"Search range: {date_label}")
        for sensor in RESULTS:
            payload = metadata_payloads[sensor]
            records = RESULTS[sensor]
            count_label = (
                f"{len(records)} grouped dates from "
                f"{payload['total_scenes']} matching scenes"
            )
            if payload["truncated"]:
                count_label += (
                    f" (showing metadata from the most recent "
                    f"{METADATA_SCENE_CAP}; narrow the search)"
                )
            print(f"\n{sensor}: {count_label}")
            table = _result_table(sensor)
            if table.empty:
                print("No imagery found.")
            else:
                display(table)


def _on_search(_):
    search_button.disabled = True
    clear_results_button.disabled = True
    stage = "reading the AOI"
    search_status.value = "<b>Status:</b> Reading the AOI..."
    try:
        query_geometry, focus_geometry = _selected_geometry()
        stage = "validating the date range"
        start_iso, end_exclusive_iso, date_label = _query_date_range()
        stage = "querying Earth Engine collections"
        search_status.value = "<b>Status:</b> Querying Earth Engine collections..."
        payloads, collections = _query_all_sensors(
            query_geometry, start_iso, end_exclusive_iso
        )
        SEARCH_CONTEXT.clear()
        SEARCH_CONTEXT.update(
            {
                "query_geometry": query_geometry,
                "focus_geometry": focus_geometry,
                "start": start_iso,
                "end_exclusive": end_exclusive_iso,
                "label": date_label,
                "collections": collections,
            }
        )

        stage = "adding the AOI to the notebook map"
        search_status.value = "<b>Status:</b> Rendering results..."
        _safe_remove_ee_layer("Query AOI")
        outline = ee.FeatureCollection(
            [ee.Feature(query_geometry)]
        ).style(color="FFFF00", fillColor="00000000", width=2)
        _add_ee_layer(outline, {}, "Query AOI")
        center_warning = ""
        try:
            _center_object(
                focus_geometry, 11 if aoi_mode.value == "point" else None
            )
        except Exception as center_error:
            center_warning = (
                " Notebook-map centering was skipped: "
                f"{html.escape(str(center_error))}"
            )

        stage = "rendering result controls and tables"
        _update_date_options()
        _update_visualization_options()
        _render_search_tables(payloads, date_label)

        total_dates = sum(len(records) for records in RESULTS.values())
        search_status.value = (
            f"<b>Status:</b> Search complete — {total_dates} grouped "
            f"sensor dates available.{center_warning}"
        )
        display_status.value = (
            "<b>Display:</b> Choose a sensor, temporal mode, and "
            "visualization."
        )
    except Exception as error:
        error_text = html.escape(str(error))
        search_status.value = (
            f"<b>Status:</b> <span style='color:#b00020'>"
            f"{type(error).__name__} during {stage}: {error_text}. "
            "See diagnostics below.</span>"
        )
        with search_output:
            clear_output(wait=True)
            print(f"Search failed during: {stage}")
            print(traceback.format_exc())
    finally:
        search_button.disabled = False
        clear_results_button.disabled = False


def _find_selected_record():
    sensor = sensor_select.value
    selected_date = date_select.value
    return next(
        (
            record
            for record in RESULTS.get(sensor, [])
            if record["date"] == selected_date
        ),
        None,
    )


def _clear_image_layers(_=None):
    for layer_name in list(DISPLAYED_LAYERS):
        _safe_remove_ee_layer(layer_name)
    DISPLAYED_LAYERS.clear()
    LAST_DISPLAY_RESULT.clear()
    _remove_arcgis_result_layers()
    display_status.value = "<b>Display:</b> Image result layers cleared."


def _on_add_layer(_):
    if not SEARCH_CONTEXT:
        display_status.value = (
            "<b>Display:</b> <span style='color:#b00020'>"
            "Search first to create an imagery collection.</span>"
        )
        return

    sensor = sensor_select.value
    preset_name = visualization_select.value
    temporal_selection = temporal_mode.value
    use_search_range = temporal_selection in {
        "range_mosaic",
        "range_reducer",
    }
    use_range_reducer = temporal_selection == "range_reducer"
    reducer_name = temporal_reducer.value if use_range_reducer else None
    reducer_label = next(
        (
            label
            for label, value in TEMPORAL_REDUCERS.items()
            if value == reducer_name
        ),
        None,
    )

    if use_search_range:
        if not RESULTS.get(sensor):
            display_status.value = (
                "<b>Display:</b> <span style='color:#b00020'>"
                f"No {html.escape(sensor)} images were returned by the "
                "current search.</span>"
            )
            return
        record = _search_range_record(sensor)
        source_collection = _search_range_collection(sensor)
        range_start, range_end = _search_range_dates()
        if temporal_selection == "range_mosaic":
            # Earth Engine mosaic gives priority to the last image. Sorting
            # oldest-to-newest therefore puts the latest acquisition on top
            # wherever multiple dates have valid pixels.
            source_collection = source_collection.sort(
                "system:time_start", True
            )
            display_status.value = (
                "<b>Display:</b> Building the search-range mosaic "
                f"for {range_start} to {range_end}..."
            )
        else:
            display_status.value = (
                f"<b>Display:</b> Building the {reducer_label.lower()} "
                f"composite for {range_start} to {range_end}..."
            )
    else:
        record = _find_selected_record()
        if record is None:
            display_status.value = (
                "<b>Display:</b> <span style='color:#b00020'>"
                "Select a returned acquisition date.</span>"
            )
            return
        source_collection = _selected_date_collection(sensor, record)
        range_start = range_end = None
        display_status.value = (
            "<b>Display:</b> Building the full-date mosaic..."
        )

    add_layer_button.disabled = True
    try:
        source_count = int(source_collection.size().getInfo())
        if source_count < 1:
            period = (
                f"{range_start} to {range_end}"
                if use_search_range
                else record["date"]
            )
            raise RuntimeError(
                f"No {sensor} granules/scenes remain for {period}."
            )

        if sensor == "Sentinel-2":
            image, vis_params = make_s2_display_image(
                record,
                preset_name,
                cloud_mask_display.value,
                source_collection=source_collection,
                temporal_reducer=reducer_name,
            )
            source_label = "granules"
        elif sensor == "Landsat":
            image, vis_params = make_landsat_display_image(
                record,
                preset_name,
                cloud_mask_display.value,
                source_collection=source_collection,
                temporal_reducer=reducer_name,
            )
            source_label = "scenes"
        else:
            image, vis_params = make_s1_display_image(
                record,
                preset_name,
                source_collection=source_collection,
                temporal_reducer=reducer_name,
            )
            source_label = "scenes"

        if replace_previous.value:
            _clear_image_layers()

        if temporal_selection == "range_mosaic":
            layer_name = (
                f"{sensor} {range_start} to {range_end} range mosaic "
                f"({source_count} {source_label}) | {preset_name}"
            )
            period_label = f"{range_start} to {range_end}"
        elif use_range_reducer:
            layer_name = (
                f"{sensor} {range_start} to {range_end} {reducer_label} "
                f"({source_count} {source_label}) | {preset_name}"
            )
            period_label = f"{range_start} to {range_end}"
        else:
            layer_name = (
                f"{sensor} {record['date']} mosaic "
                f"({source_count} {source_label}) | {preset_name}"
            )
            period_label = record["date"]

        _safe_remove_ee_layer(layer_name)
        _add_ee_layer(image, vis_params, layer_name, True)
        if layer_name not in DISPLAYED_LAYERS:
            DISPLAYED_LAYERS.append(layer_name)

        focus_geometry = SEARCH_CONTEXT.get("query_geometry")
        LAST_DISPLAY_RESULT.clear()
        LAST_DISPLAY_RESULT.update(
            {
                "image": image,
                "vis_params": vis_params,
                "layer_name": layer_name,
                "focus_geometry": focus_geometry,
                "sensor": sensor,
                "period": period_label,
                "temporal_mode": temporal_selection,
                "reducer": reducer_name,
                "source_count": source_count,
                "source_label": source_label,
            }
        )
        center_message = ""
        if zoom_to_result.value and focus_geometry is not None:
            try:
                _center_object(focus_geometry)
            except Exception as center_error:
                center_message = (
                    " Notebook-map centering was skipped: "
                    f"{html.escape(str(center_error))}."
                )

        arcgis_message = ""
        if mirror_to_arcgis.value:
            try:
                target_name = _arcgis_add_layer(layer_name, focus_geometry)
                arcgis_message = f" Mirrored to {target_name}."
                arcgis_status.value = (
                    f"<b>ArcGIS Pro:</b> Added {html.escape(layer_name)} "
                    f"to {html.escape(target_name)}."
                )
            except Exception as arcgis_error:
                if NOTEBOOK_IO_LOOP is None:
                    arcgis_status.value = (
                        "<b>ArcGIS Pro:</b> Result prepared. Run the "
                        "<b>Send the prepared layer to ArcGIS Pro</b> code "
                        "cell below the controls."
                    )
                    arcgis_message = (
                        " ArcGIS Pro transfer is ready in the code cell below."
                    )
                else:
                    arcgis_status.value = (
                        f"<b>ArcGIS Pro:</b> <span style='color:#b00020'>"
                        f"{html.escape(str(arcgis_error))}</span>"
                    )
                    arcgis_message = (
                        " ArcGIS mirroring failed; see its status below."
                    )

        if temporal_selection == "range_mosaic":
            result_description = (
                f"Added a date-range mosaic of all {source_count} "
                f"{source_label} from {range_start} to {range_end}; "
                "the latest valid acquisition has overlap priority."
            )
        elif use_range_reducer:
            result_description = (
                f"Added the {reducer_label.lower()} composite of all "
                f"{source_count} {source_label} from {range_start} "
                f"to {range_end}."
            )
        else:
            result_description = (
                f"Added a mosaic of all {source_count} {source_label} "
                f"for {record['date']}."
            )
        display_status.value = (
            f"<b>Display:</b> {result_description}"
            f"{center_message}{arcgis_message}"
        )
    except Exception as error:
        display_status.value = (
            f"<b>Display:</b> <span style='color:#b00020'>"
            f"{type(error).__name__}: {html.escape(str(error))}</span>"
        )
    finally:
        add_layer_button.disabled = False


def _on_clear_results(_):
    _clear_image_layers()
    for sensor in RESULTS:
        RESULTS[sensor] = []
    SEARCH_CONTEXT.clear()
    date_select.options = []
    with search_output:
        clear_output(wait=True)
    search_status.value = "<b>Status:</b> Results cleared; AOI retained."
    display_status.value = "<b>Display:</b> Search first."


# Populate map/scene choices now, while this cell is on the notebook thread.
if ARCGIS_CONTEXT["project"] is not None:
    try:
        initial_arcgis_targets = _refresh_arcgis_targets()
        _apply_arcgis_targets(initial_arcgis_targets)
        arcgis_status.value = (
            "<b>ArcGIS Pro:</b> Ready — the current project was captured "
            "during cell execution."
        )
    except Exception as error:
        arcgis_status.value = (
            "<b>ArcGIS Pro:</b> <span style='color:#b00020'>"
            f"{html.escape(str(error))}</span>"
        )

sensor_select.observe(_on_sensor_change, names="value")
temporal_mode.observe(_sync_temporal_controls, names="value")
_sync_temporal_controls()
search_button.on_click(_on_search)
clear_results_button.on_click(_on_clear_results)
add_layer_button.on_click(_on_add_layer)
clear_layers_button.on_click(_clear_image_layers)
test_arcgis_button.on_click(_on_test_arcgis)

search_controls = widgets.VBox(
    [
        widgets.HTML("<h3>Search</h3>"),
        widgets.HBox([buffer_km, cloud_max, max_dates]),
        widgets.HBox([use_custom_dates, lookback_days]),
        widgets.HBox([start_date, end_date]),
        widgets.HBox([s2_level, landsat_level, s1_mode]),
        widgets.HBox([search_button, clear_results_button]),
        search_status,
    ]
)
display_controls = widgets.VBox(
    [
        widgets.HTML("<h3>Display a result</h3>"),
        widgets.HBox([sensor_select, date_select]),
        widgets.HBox([temporal_mode, temporal_reducer]),
        widgets.HTML(
            "<small>Range mosaic uses all filtered images with the latest "
            "valid pixel on top; range reducers aggregate the same complete "
            "search collection.</small>"
        ),
        widgets.HBox([visualization_select, cloud_mask_display]),
        widgets.HBox([replace_previous, zoom_to_result]),
        widgets.HBox([mirror_to_arcgis, open_arcgis_view]),
        widgets.HBox([test_arcgis_button, arcgis_target]),
        arcgis_status,
        widgets.HBox([add_layer_button, clear_layers_button]),
        display_status,
    ]
)

display(search_controls, search_output, display_controls)


## 7. Send the prepared layer to ArcGIS Pro

ArcGIS Pro runs widget clicks on a background thread in some notebook
versions. That thread cannot use `ArcGISProject("CURRENT")`. The older
`0_General_s1s2.ipynb` worked because `Map.addLayer(...)` was executed in a
normal code cell.

Inside ArcGIS Pro, first use the controls above to search and click **Add
selected image**. Then rerun the following code cell. It performs the same
underlying operation as geemap's ArcGIS-aware `Map.addLayer()` path—adding the
Earth Engine web-tile URL to the selected Pro map or scene—but from the
application-safe notebook execution context.

In [ ]:
# ArcGIS Pro only: rerun this cell after preparing a result above.
if LAST_DISPLAY_RESULT:
    arcgis_target_label = add_prepared_result_to_arcgis()
    print(f"Added the prepared mosaic to {arcgis_target_label}.")
else:
    print(
        "No mosaic is prepared yet. Search, choose an output, click "
        "'Add selected image', and then rerun this cell."
    )

## 8. Checks, limitations, and next steps

- Results are grouped by **UTC acquisition date** and mosaicked in the
  returned collection order, matching the central behavior of the source script.
- Sentinel-2 and Landsat cloud percentages filter collection metadata.
  Optional QA/SCL masking is applied only when drawing a layer.
- Sentinel-1 dual-polarization visualizations automatically use VV/VH or
  HH/HV. A requested unavailable single band falls back to the preferred
  available band.
- QuickFire is a visualization aid transferred from the source script,
  not a calibrated cross-sensor fire-detection product.
- Very large AOIs or date ranges can be expensive. If the 2,500-scene
  metadata cap is reached, reduce the AOI, date range, or cloud threshold.
- A selected date is rebuilt from the complete server-side filtered
  collection and mosaicked, so the layer includes every intersecting granule or
  scene for that UTC date. The displayed status reports the server-side count.
- The optional **Search-range mosaic** uses every filtered image from
  the current search interval without a statistical reducer. Images are
  ordered oldest-to-newest so the latest valid pixel has priority where dates
  overlap; masked gaps can be filled by older acquisitions.
- The optional **Search-range reducer** uses the complete server-side
  collection from the current search interval—not only the dates shown in the
  dropdown—and supports minimum, mean, median, maximum, and mode. Optical
  presets are prepared per image before reduction; Sentinel-1 bands are
  reduced before its selected radar visualization/index is calculated.
- ArcGIS mirroring is optional and requires the notebook to run inside the
  ArcGIS Pro application. Some Pro releases do not expose a kernel event
  loop to widget callbacks. In that case the button prepares the mosaic;
  rerun the **Send the prepared layer to ArcGIS Pro** code cell to add it
  from the application-safe notebook execution context. External VS
  Code/Jupyter kernels cannot alter an already-open ArcGIS scene.
- ArcGIS transfer never calls `Map.openView()`, because Pro creates a new
  tab even if that Scene already has one. The existing Scene receives the
  layer directly. The optional camera checkbox adjusts the view only when
  that target Scene is already the active ArcGIS view.
- In a 3D scene, the Earth Engine web tile layer is draped over the scene's
  ground/elevation surface; it is not converted into a volumetric 3D dataset.
- Notebook-map navigation sets center/zoom synchronously rather than calling
  ipyleaflet's asynchronous `fit_bounds()`. This avoids ArcGIS Pro background
  widget-thread errors such as `There is no current event loop in thread`.
- The notebook intentionally does not implement the JavaScript tool's
  water detection, hotspot inspection, pixel inspector, or export panels.

Before committing future notebook changes, restart the kernel, run cells
in order, inspect for identifiers or private data, and clear all outputs.
Never save credentials inside notebook cells or outputs.
